In [72]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import h5py

In [73]:
def gen_spikes(rate_tensor):
    random_vals = torch.rand_like(rate_tensor)
    spikes = (random_vals < rate_tensor).float()
    
    return spikes

In [74]:
class ATan(torch.autograd.Function):
    
    @staticmethod
    def forward(ctx, input_, alpha):
        ctx.save_for_backward(input_)
        ctx.alpha = alpha
        out = (input_ > 0).float()
        return out

    @staticmethod
    def backward(ctx, grad_output):
        (input_,) = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad = (
            ctx.alpha
            / 2
            / (1 + (torch.pi / 2 * ctx.alpha * input_).pow_(2))
            * grad_input
        )
        return grad, None


class LIF:


    def __init__(self, time_step, R, C, V_threshold):

        self.time_step = time_step
        self.threshold = V_threshold
        self.R = R
        self.C = C
        self.tau = R * C

    def __call__(self, *args, **kwds):
        return self.forward(*args, **kwds)

    # neuron function (non-leaky)
    def forward(self, I, V_prev):

        # fire and reset mechanisms
        #spk_out = (V_prev >= self.threshold).float()
        spk_out = ATan.apply(V_prev - self.threshold, 25.0)

        V = V_prev + (self.time_step / self.tau) * (-V_prev + self.R * I) - spk_out * self.threshold

        return spk_out, V

In [75]:
class Net(nn.Module):
    # define all the constants
    R_m = 5.1  # Membrane resistance in MΩ
    C_m = 5e-3  # Membrane capacitance in pF
    time_step = 1e-3  # Time step in ms

    V_threshold = 1  # Threshold potential for firing a spike in mV


    hidden_features = 128

    def __init__(self, timesteps, in_features, out_features):
        super(Net, self).__init__()

        self.timesteps = timesteps
        self.in_features = in_features
        self.out_features = out_features

        self.fc1 = nn.Linear(in_features, self.hidden_features)
        self.fc2 = nn.Linear(self.hidden_features, out_features)

        self.lif1 = LIF(time_step=self.time_step, R=self.R_m, C=self.C_m, V_threshold=self.V_threshold)
        self.lif2 = LIF(time_step=self.time_step, R=self.R_m, C=self.C_m, V_threshold=self.V_threshold)


    def forward(self, x):
        #x = x.view(-1, 28 * 28)  # Flatten
        
        # initialize membrane potentials with [batch_size, num_neurons]
        mem1 = torch.zeros(x.size(0), self.hidden_features)
        mem2 = torch.zeros(x.size(0), self.out_features)

        # Record the spikes from the last layer
        spk1_rec = []
        mem1_rec = []
        spk2_rec = []
        mem2_rec = []
        
        for t in range(self.timesteps):

            cur1 = self.fc1(x) # in : [batch_size, in_features] out : [batch_size, hidden_features]
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            
            spk1_rec.append(spk1)
            mem1_rec.append(mem1)
            spk2_rec.append(spk2)
            mem2_rec.append(mem2)
        

        return torch.stack(spk2_rec, dim=1), torch.stack(mem2_rec, dim=1), torch.stack(spk1_rec, dim=1), torch.stack(mem1_rec, dim=1)

In [76]:
def save_snn_recording(input_raw, spikes_hidden, membrane_hidden, 
                       spikes_output, membrane_output, targets, predictions,
                       epochs, iterations, filename):

    
    # Convert to numpy
    input_raw = input_raw.numpy()
    spikes_hidden = spikes_hidden.numpy()
    membrane_hidden = membrane_hidden.numpy()
    spikes_output = spikes_output.numpy()
    membrane_output = membrane_output.numpy()
    targets = targets.numpy()
    predictions = predictions.numpy()
    
    # Get dimensions
    num_iterations = input_raw.shape[0]
    batch_size = input_raw.shape[1]
    num_timesteps = spikes_hidden.shape[2]
    hidden_size = spikes_hidden.shape[3]
    output_size = spikes_output.shape[3]
    
    print(f"Saving {num_iterations} iterations, batch_size={batch_size}...")
    
    # Save to HDF5
    with h5py.File(filename, 'w') as f:
        # Metadata
        meta = f.create_group('metadata')
        meta.create_dataset('num_iterations', data=num_iterations)
        meta.create_dataset('batch_size', data=batch_size)
        meta.create_dataset('num_timesteps', data=num_timesteps)
        meta.create_dataset('layer_sizes', data=np.array([784, hidden_size, output_size]))
        meta.create_dataset('num_layers', data=2)
        meta.create_dataset('dt', data=1.0)
        meta.create_dataset('epochs', data=np.array(epochs))
        meta.create_dataset('iterations', data=np.array(iterations))
        
        # Input data
        f.create_dataset('input_raw', data=input_raw, compression='gzip')
        f.create_dataset('targets', data=targets, compression='gzip')
        f.create_dataset('predictions', data=predictions, compression='gzip')  # NEW!
        
        # Hidden layer
        f.create_dataset('spikes_layer_1', 
                       data=spikes_hidden.astype(np.uint8), 
                       compression='gzip')
        f.create_dataset('membrane_layer_1', 
                       data=membrane_hidden.astype(np.float32), 
                       compression='gzip')
        
        # Output layer
        f.create_dataset('spikes_layer_2', 
                       data=spikes_output.astype(np.uint8), 
                       compression='gzip')
        f.create_dataset('membrane_layer_2', 
                       data=membrane_output.astype(np.float32), 
                       compression='gzip')
    
    print(f"Saved to {filename}")

In [77]:
learning_rate = 0.001

num_steps = 25
in_features = 5
out_features = 10
num_batches = 1

num_epochs = 1

# Load data
transform = transforms.ToTensor()
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)


In [78]:

model = Net(timesteps=num_steps, in_features=784, out_features=10)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()


In [79]:
# single forward pass, validate network and inspect shapes
with torch.no_grad():
    total = 0
    correct = 0
    model.eval()

    
    for data, targets in test_loader:
        print(f"Input shape: {data.shape}") # Should be [batch_size, 1, 28, 28]

        data = data.view(-1, 28 * 28)  # Flatten

        print(f"Flattened input shape: {data.shape}") # Should be [batch_size, 784]
        
        spk_out, mem_out, spk1, mem1 = model(data)

        print(f"Output spike tensor shape: {spk_out.shape}")  # Should be [batch_size, timesteps, out_features]
        print(f"Output membrane potential tensor shape: {mem_out.shape}")  # Should be [batch_size, timesteps, out_features]
        print(f"Hidden layer spike tensor shape: {spk1.shape}")  # Should be [batch_size, timesteps, hidden_features]
        print(f"Hidden layer membrane potential tensor shape: {mem1.shape}")  # Should be [batch_size, timesteps, hidden_features]


        spike_counts = spk_out.sum(dim=1)

        print(f"Spike counts shape: {spike_counts.shape}")  # Should be [batch_size, out_features]

        predictions = spike_counts.argmax(dim=1)

        print(f"Predictions shape: {predictions.shape}")  # Should be [batch_size]
        break

Input shape: torch.Size([1000, 1, 28, 28])
Flattened input shape: torch.Size([1000, 784])
Output spike tensor shape: torch.Size([1000, 25, 10])
Output membrane potential tensor shape: torch.Size([1000, 25, 10])
Hidden layer spike tensor shape: torch.Size([1000, 25, 128])
Hidden layer membrane potential tensor shape: torch.Size([1000, 25, 128])
Spike counts shape: torch.Size([1000, 10])
Predictions shape: torch.Size([1000])


In [ ]:
loss_hist = [] # record loss over iterations
acc_hist = [] # record accuracy over iterations

# Initialize collection lists
collected_data = {
    'input_raw': [],
    'spikes_hidden': [],
    'membrane_hidden': [],
    'spikes_output': [],
    'membrane_output': [],
    'targets': [],
    'predictions': [],
    'epochs': [],
    'iterations': []
}

collect_data = True
iteration_counter = 0

for epoch in range(num_epochs):
    for i, (data, targets) in enumerate(train_loader):

        data = data.view(-1, 28 * 28)  # Flatten

        model.train()

        spk_out, mem_out, spk1, mem1 = model(data)
        
        logits = spk_out.sum(dim=1)

        predictions = logits.argmax(dim=1)

        loss = nn.CrossEntropyLoss()(logits, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_hist.append(loss.item())

        # Collect data
        if collect_data:
            collected_data['input_raw'].append(data.detach().cpu())
            collected_data['spikes_hidden'].append(spk1.detach().cpu())
            collected_data['membrane_hidden'].append(mem1.detach().cpu())
            collected_data['spikes_output'].append(spk_out.detach().cpu())
            collected_data['membrane_output'].append(mem_out.detach().cpu())
            collected_data['targets'].append(targets.detach().cpu())
            collected_data['predictions'].append(predictions.detach().cpu())
            collected_data['epochs'].append(epoch)
            collected_data['iterations'].append(iteration_counter)
    
        iteration_counter += 1

        # print every 25 iterations
        if i % 25 == 0:
            model.eval()
            print(f"Epoch {epoch}, Iteration {i} \nTrain Loss: {loss.item():.2f}")
            

            correct += (predictions == targets).sum().item()
            total += targets.size(0)
        
            acc = correct / total
            acc_hist.append(acc)
            print(f"Accuracy: {acc * 100:.2f}%\n")

if collect_data:
    print("\nCombining and saving collected data...")
    save_snn_recording(
        input_raw=torch.stack(collected_data['input_raw'], dim=0),
        spikes_hidden=torch.stack(collected_data['spikes_hidden'], dim=0),
        membrane_hidden=torch.stack(collected_data['membrane_hidden'], dim=0),
        spikes_output=torch.stack(collected_data['spikes_output'], dim=0),
        membrane_output=torch.stack(collected_data['membrane_output'], dim=0),
        targets=torch.stack(collected_data['targets'], dim=0),
        predictions=torch.stack(collected_data['predictions'], dim=0),
        epochs=collected_data['epochs'],
        iterations=collected_data['iterations'],
        filename="snn_recording.h5"
    )
print("Data saved")

Epoch 0, Iteration 0 
Train Loss: 1.95
Accuracy: 29.85%

Epoch 0, Iteration 25 
Train Loss: 2.12
Accuracy: 29.84%

Epoch 0, Iteration 50 
Train Loss: 2.11
Accuracy: 29.76%

Epoch 0, Iteration 75 
Train Loss: 1.97
Accuracy: 29.69%

Epoch 0, Iteration 100 
Train Loss: 1.93
Accuracy: 29.76%

Epoch 0, Iteration 125 
Train Loss: 2.10
Accuracy: 29.65%

Epoch 0, Iteration 150 
Train Loss: 2.00
Accuracy: 29.69%

Epoch 0, Iteration 175 
Train Loss: 2.13
Accuracy: 29.72%

Epoch 0, Iteration 200 
Train Loss: 2.01
Accuracy: 29.79%

Epoch 0, Iteration 225 
Train Loss: 1.95
Accuracy: 29.75%

Epoch 0, Iteration 250 
Train Loss: 1.94
Accuracy: 29.75%

Epoch 0, Iteration 275 
Train Loss: 1.95
Accuracy: 29.81%

Epoch 0, Iteration 300 
Train Loss: 1.91
Accuracy: 30.02%

Epoch 0, Iteration 325 
Train Loss: 2.13
Accuracy: 29.96%

Epoch 0, Iteration 350 
Train Loss: 2.03
Accuracy: 29.81%

Epoch 0, Iteration 375 
Train Loss: 2.11
Accuracy: 29.77%

Epoch 0, Iteration 400 
Train Loss: 1.95
Accuracy: 29.80%

Ep

: 

In [32]:
with torch.no_grad():
    total = 0
    correct = 0
    model.eval()

    for data, targets in test_loader:
        data = data.view(-1, 28 * 28)  # Flatten
        
        spk_out, _ = model(data)

        spike_counts = spk_out.sum(dim=1)
        predictions = spike_counts.argmax(dim=1)

        correct += (predictions == targets).sum().item()
        total += targets.size(0)

    print(f"Accuracy: {100 * correct / total:.2f}%")

Accuracy: 11.21%
